# Web attack detection - clean notebook
Notebook này chỉ đọc dữ liệu trong `dataset/raw_data`, xử lý in-memory, train/evaluate model và hiển thị kết quả. Không tải thêm dữ liệu, không ghi file mới, không export artifact.

## Quy ước ngắn
- Bài toán mặc định: phân loại nhị phân `Normal` vs `Attack`
- Chọn model bằng validation set
- Test set chỉ dùng đúng 1 lần để báo cáo cuối

In [ ]:
from pathlib import Path
import hashlib
import re

import matplotlib.pyplot as plt
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC

SEED = 42
PROJECT_ROOT = Path.cwd()
DATA_ROOT = PROJECT_ROOT / "dataset" / "raw_data"

if not DATA_ROOT.exists():
    raise FileNotFoundError(f"Không tìm thấy thư mục dữ liệu: {DATA_ROOT}")

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_ROOT:", DATA_ROOT)

In [ ]:
TEXT_COLUMN_CANDIDATES = [
    "text", "sentence", "payload", "query", "request", "url", "content", "data", "input", "value"
]
LABEL_COLUMN_CANDIDATES = [
    "label", "class", "target", "attack", "type", "result", "is_attack"
]

NORMAL_VALUES = {"0", "normal", "benign", "safe", "clean", "false", "norm", "legitimate"}
ATTACK_VALUES = {"1", "attack", "malicious", "true", "anomaly", "anomalous"}

CONTROL_RE = re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]")
MULTISPACE_RE = re.compile(r"\s+")


def sha256_text(value: str) -> str:
    return hashlib.sha256(value.encode("utf-8", errors="ignore")).hexdigest()


def normalize_text(value) -> str:
    value = "" if value is None else str(value)
    value = value.replace("\x00", " ")
    value = CONTROL_RE.sub(" ", value)
    value = value.replace("\r", " ").replace("\n", " ").replace("\t", " ")
    value = MULTISPACE_RE.sub(" ", value).strip().lower()
    return value


def clean_column_name(name: str) -> str:
    name = str(name).replace("\ufeff", "").replace("\x00", "").strip().lower()
    name = re.sub(r"[^a-z0-9]+", "_", name).strip("_")
    return name or "unnamed"


def make_unique(columns):
    counts = {}
    unique = []
    for col in columns:
        count = counts.get(col, 0)
        unique.append(col if count == 0 else f"{col}_{count}")
        counts[col] = count + 1
    return unique


def clean_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = make_unique([clean_column_name(c) for c in df.columns])
    for col in df.columns:
        if df[col].dtype == "object":
            df[col] = df[col].map(lambda x: x if pd.isna(x) else str(x).replace("\x00", "").strip())
    return df


def read_csv_robust(path: Path) -> pd.DataFrame:
    attempts = [
        {"encoding": "utf-8-sig", "sep": ","},
        {"encoding": "utf-16", "sep": ","},
        {"encoding": "utf-16-le", "sep": ","},
        {"encoding": "utf-16-be", "sep": ","},
        {"encoding": "latin1", "sep": ","},
        {"encoding": "utf-8-sig", "sep": "\t"},
        {"encoding": "latin1", "sep": "\t"},
    ]
    last_error = None
    for attempt in attempts:
        try:
            df = pd.read_csv(path, encoding=attempt["encoding"], sep=attempt["sep"], engine="python")
            return clean_dataframe(df)
        except Exception as exc:
            last_error = exc
    raise RuntimeError(f"Không đọc được {path.name}: {last_error}")


def find_first(columns, candidates):
    for candidate in candidates:
        if candidate in columns:
            return candidate
    return None


def parse_binary_label(value, *, default=None):
    if pd.isna(value):
        return default
    value = normalize_text(value)
    if value in NORMAL_VALUES:
        return 0
    if value in ATTACK_VALUES:
        return 1
    if "normal" in value or "benign" in value:
        return 0
    if any(keyword in value for keyword in ["sqli", "sql", "xss", "attack", "inject", "traversal", "command"]):
        return 1
    return default


def summarize_rows(df: pd.DataFrame, name: str):
    print(f"{name}: {len(df):,} rows")
    if not df.empty:
        print(df[["source", "label_name"]].value_counts().head(10))
        print()

In [ ]:
def load_csic_dataset(path: Path) -> pd.DataFrame:
    df = read_csv_robust(path)
    if df.empty:
        return pd.DataFrame(columns=["source", "raw_text", "label", "label_name"])

    label_col = df.columns[0]
    text_cols = [col for col in df.columns[1:] if not col.startswith("unnamed")]
    if not text_cols:
        raise ValueError("CSIC dataset không có cột nội dung để ghép request")

    records = []
    for _, row in df.iterrows():
        label = parse_binary_label(row[label_col], default=None)
        if label is None:
            continue

        parts = []
        for col in text_cols:
            value = row[col]
            if pd.isna(value):
                continue
            value = str(value).strip()
            if not value or value.lower() == "nan":
                continue
            parts.append(f"{col}={value}")

        raw_text = " | ".join(parts)
        raw_text = normalize_text(raw_text)
        if not raw_text:
            continue

        records.append({
            "source": "CSIC",
            "raw_text": raw_text,
            "label": label,
            "label_name": "Normal" if label == 0 else "Attack",
        })

    return pd.DataFrame(records)


def load_payload_dataset(path: Path, source: str, attack_name: str) -> pd.DataFrame:
    df = read_csv_robust(path)
    text_col = find_first(df.columns, TEXT_COLUMN_CANDIDATES)
    label_col = find_first(df.columns, LABEL_COLUMN_CANDIDATES)

    if text_col is None:
        raise ValueError(f"{path.name}: không tìm thấy text column sau khi làm sạch header: {list(df.columns)}")

    records = []
    for _, row in df.iterrows():
        raw_text = normalize_text(row[text_col])
        if not raw_text or raw_text == "nan":
            continue

        label = parse_binary_label(row[label_col], default=1) if label_col else 1
        records.append({
            "source": source,
            "raw_text": raw_text,
            "label": label,
            "label_name": "Normal" if label == 0 else attack_name,
        })

    return pd.DataFrame(records)


MANIFEST = [
    {"kind": "csic", "path": DATA_ROOT / "http-CSIC-2010" / "csic_database.csv"},
    {"kind": "payload", "path": DATA_ROOT / "sqlinjectionextend" / "sqli-extended.csv", "source": "SQLI_EXTENDED", "attack_name": "SQLi"},
    {"kind": "payload", "path": DATA_ROOT / "sqli" / "sqli.csv", "source": "SQLI", "attack_name": "SQLi"},
    {"kind": "payload", "path": DATA_ROOT / "sqli" / "sqliv2.csv", "source": "SQLI_V2", "attack_name": "SQLi"},
    {"kind": "payload", "path": DATA_ROOT / "sqli" / "SQLiV3.csv", "source": "SQLI_V3", "attack_name": "SQLi"},
    {"kind": "payload", "path": DATA_ROOT / "xss" / "XSS_dataset.csv", "source": "XSS", "attack_name": "XSS"},
]

frames = []
for item in MANIFEST:
    path = item["path"]
    if not path.exists():
        print("Bỏ qua file không tồn tại:", path)
        continue

    if item["kind"] == "csic":
        frame = load_csic_dataset(path)
    else:
        frame = load_payload_dataset(path, item["source"], item["attack_name"])

    summarize_rows(frame, path.name)
    frames.append(frame)

if not frames:
    raise RuntimeError("Không có dataset nào được load thành công")

data = pd.concat(frames, ignore_index=True)
print("Tổng số dòng trước cleaning:", f"{len(data):,}")

In [ ]:
data = data.copy()
data["normalized_text"] = data["raw_text"].map(normalize_text)
data = data[data["normalized_text"].str.len() > 0].copy()
data["text_hash"] = data["normalized_text"].map(sha256_text)

conflict_hashes = (
    data.groupby("text_hash")["label"]
    .nunique()
    .loc[lambda s: s > 1]
    .index
)

before_conflict = len(data)
data = data[~data["text_hash"].isin(conflict_hashes)].copy()
after_conflict = len(data)

before_dedup = len(data)
data = data.drop_duplicates(subset="text_hash", keep="first").reset_index(drop=True)
after_dedup = len(data)

print("Rows after cleaning:", f"{len(data):,}")
print("Removed by label conflict:", before_conflict - after_conflict)
print("Removed by dedup:", before_dedup - after_dedup)
print()
print(data[["source", "label_name"]].value_counts())
print()

dataset_summary = (
    data.groupby("source")
    .agg(
        rows=("source", "size"),
        normal=("label", lambda s: int((s == 0).sum())),
        attack=("label", lambda s: int((s == 1).sum())),
    )
    .sort_values("rows", ascending=False)
)
print(dataset_summary)

## Train / validation / test split
Dữ liệu được split theo `label` để chọn model bằng validation trước, sau đó mới đánh giá test một lần.

In [ ]:
if data["label"].nunique() < 2:
    raise RuntimeError("Dataset hiện tại không đủ 2 lớp để train classifier")

train_df, temp_df = train_test_split(
    data,
    test_size=0.30,
    random_state=SEED,
    stratify=data["label"],
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_df["label"],
)

for name, frame in [("train", train_df), ("validation", val_df), ("test", test_df)]:
    print(name, len(frame), frame["label"].value_counts().to_dict())

split_overlap = {
    "train_val": len(set(train_df["text_hash"]) & set(val_df["text_hash"])),
    "train_test": len(set(train_df["text_hash"]) & set(test_df["text_hash"])),
    "val_test": len(set(val_df["text_hash"]) & set(test_df["text_hash"])),
}
print("Hash overlap:", split_overlap)
assert split_overlap == {"train_val": 0, "train_test": 0, "val_test": 0}

In [ ]:
def evaluate_frame(model, frame: pd.DataFrame, split_name: str) -> dict:
    y_true = frame["label"]
    y_pred = model.predict(frame["normalized_text"])
    return {
        "split": split_name,
        "accuracy": accuracy_score(y_true, y_pred),
        "f1_macro": f1_score(y_true, y_pred, average="macro"),
        "f1_weighted": f1_score(y_true, y_pred, average="weighted"),
    }


models = {
    "logreg_char_tfidf": Pipeline([
        ("tfidf", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=2, max_features=100_000)),
        ("clf", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=SEED)),
    ]),
    "linearsvc_char_tfidf": Pipeline([
        ("tfidf", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=2, max_features=100_000)),
        ("clf", LinearSVC(class_weight="balanced", random_state=SEED)),
    ]),
}

validation_results = []
for name, model in models.items():
    model.fit(train_df["normalized_text"], train_df["label"])
    validation_results.append({"model": name, **evaluate_frame(model, val_df, "validation")})

validation_table = pd.DataFrame(validation_results).sort_values(["f1_macro", "accuracy"], ascending=False)
print(validation_table)

best_model_name = validation_table.iloc[0]["model"]
print("Best model on validation:", best_model_name)

In [ ]:
best_model = models[best_model_name]
trainval_df = pd.concat([train_df, val_df], ignore_index=True)
best_model.fit(trainval_df["normalized_text"], trainval_df["label"])

y_test = test_df["label"]
y_pred = best_model.predict(test_df["normalized_text"])

print("Final test report for:", best_model_name)
print()
print(classification_report(y_test, y_pred, target_names=["Normal", "Attack"], zero_division=0))

cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Normal", "Attack"]).plot(cmap="Blues")
plt.title(f"Test confusion matrix - {best_model_name}")
plt.show()